# Урок 4.4. Оптимизация поиска: reranking и multi-query

В уроке 4.1 мы поставили диагноз: на вопрос «Можно ли работать из Грузии два
месяца?» наш поиск приносил `guide-onboarding` вместо `policy-remote`, потому
что в документе написано «из-за границы», а не «Грузия». Я обещал, что мы это
**починим**. Сегодня чиним — и попутно разбираемся, почему одного хорошего
эмбеддера мало.

> **Идея урока:** качественный поиск — это не «одна волшебная модель», а
> **конвейер из двух ступеней**: сначала широким неводом собрать всё, что
> *может* подойти (recall), потом точным инструментом выбрать лучшее (precision).

## 🎯 Цели урока
- Понять баланс **recall vs precision** и зачем нужен двухступенчатый поиск.
- Собрать **гибридный** поиск: плотный (смысл) + BM25 (слова) со слиянием **RRF**.
- Применить **reranking** — и увидеть вживую, как он вытаскивает `policy-remote`
  с задворков выдачи на первое место.
- Разобрать **multi-query** (переформулировка запроса через LLM) — и честно
  увидеть его пределы на маленькой модели.
- Понять, где нужен **семантический эмбеддер** (e5), а где хватает лексики.


## Recall и precision: две разные задачи

```
RECALL (полнота)          PRECISION (точность)
«ничего не потерять»      «наверх — только лучшее»
широкий невод             острый фильтр
дёшево, много кандидатов  дорого, мало кандидатов
   dense + BM25 + RRF          reranking
        ↓                          ↓
   200 кандидатов  ───────▶   top-4 в промпт
```

Одна модель редко хороша в обоих. Быстрый поиск (эмбеддинги, BM25) оптимизирован
под recall: пробежать миллион чанков за миллисекунды и не потерять нужный.
Точная модель (cross-encoder) оптимизирована под precision: прочитать пару
(вопрос, чанк) целиком и честно оценить — но она слишком медленная, чтобы
прогонять её по всему индексу.

Отсюда **двухступенчатая схема**: широкий невод отбирает 20–200 кандидатов,
дорогой reranker переупорядочивает их и оставляет 4 лучших. Именно так устроен
`retrieve()` в нашем `rag_service`.


In [ ]:
import sys
import os
from pathlib import Path

MODULE_DIR = Path.cwd()
print(f"Current working directory: {MODULE_DIR}")
if not (MODULE_DIR / "rag_service").exists() and (MODULE_DIR.parent / "rag_service").exists():
    MODULE_DIR = MODULE_DIR.parent
    print(f"Adjusted working directory to: {MODULE_DIR}")
sys.path.insert(0, str(MODULE_DIR))
os.environ.setdefault("RAG_JSON_LOGS", "0")

In [ ]:
from rag_service.store import KnowledgeBase
from rag_service.embeddings import get_embedder
from rag_service.llm import OllamaLLM
from rag_service.access import DEMO_PRINCIPALS
from rag_service.retrieval import LexicalIndex, retrieve, format_hits
from rag_service.observability import configure_logging

configure_logging("ERROR", json_logs=False)
kb = KnowledgeBase()
QDRANT_UP = kb.available() and kb.collection_exists()
OLLAMA_UP = OllamaLLM().available()
if QDRANT_UP:
    emb = get_embedder()
    lex = LexicalIndex.from_kb(kb)
    employee = DEMO_PRINCIPALS["demo-employee-token"]
print(f"Qdrant: {'✓' if QDRANT_UP else '✗'} | Ollama: {'✓' if OLLAMA_UP else '✗'}")

## Ступень 1. Гибридный поиск: dense + BM25

У каждого способа поиска — свои слепые зоны (мы видели это в уроке 4.1):

- **Плотный поиск (эмбеддинги)** понимает смысл, но слеп к точным строкам,
  кодам, редкому жаргону: «10.10.0.53» для него — просто набор цифр.
- **BM25 (лексический)** ловит точные слова и коды, но не видит синонимов:
  «Грузия» и «из-за границы» для него — разные вселенные.

Проверим обе слабости на одном стенде.

In [ ]:
if QDRANT_UP:
    print("=== Точная строка '10.10.0.53' ===")
    r_dense = retrieve("10.10.0.53", employee, kb, emb, lexical=lex, use_bm25=False)
    r_hyb = retrieve("10.10.0.53", employee, kb, emb, lexical=lex, use_bm25=True)
    print("только dense — top-1:", r_dense.hits[0].doc_id if r_dense.hits else "—",
          "| sim:", r_dense.hits[0].similarity if r_dense.hits else "—")
    print("dense+BM25   — top-1:", r_hyb.hits[0].doc_id if r_hyb.hits else "—",
          "| bm25:", r_hyb.hits[0].bm25_score if r_hyb.hits else "—")
    print("\nТочную строку вытягивает именно BM25 — плотный поиск по цифрам почти бесполезен.")
else:
    print("Мой прогон: '10.10.0.53' находит guide-vpn через BM25 (bm25≈2.9),")
    print("плотный косинус по цифрам низкий. Точные совпадения — работа BM25.")

### RRF: как слить два списка

Проблема слияния: косинус `0.71` и BM25 `12.4` — величины из разных вселенных,
честно нормализовать их нельзя. Решение — **Reciprocal Rank Fusion**: складываем
не скоры, а **ранги**.

```
RRF_score(чанк) = Σ  weight / (k + rank_в_списке)     k = 60
              по всем спискам
```

Ранги сравнимы всегда: 1-е место в dense и 1-е в BM25 весят одинаково.
Константа `k=60` (из статьи Cormack et al., 2009) гасит разницу между 1-м и 2-м
местом, чтобы один список не забивал другой. Чанк, попавший в топ **обоих**
списков, всплывает наверх — это и есть сила гибрида.

In [ ]:
from rag_service.retrieval import rrf_fuse

if QDRANT_UP:
    # смоделируем два коротких прогона руками, чтобы увидеть арифметику RRF
    q = "оборудование для удалённой работы"
    flt = None
    from rag_service.access import build_filter
    flt = build_filter(employee)
    dense = kb.search(emb.encode_query(q), k=5, flt=flt)
    bm = lex.search(q, k=5)
    print("dense top-5:", [h.doc_id for h in dense])
    print("bm25  top-5:", [h.doc_id for h in bm])
    fused = rrf_fuse({"dense": dense, "bm25": bm})
    print("\nпосле RRF (по убыванию rrf_score):")
    for h in fused[:5]:
        print(f"  {h.doc_id:20s} rrf={h.rrf_score}  ранги={h.ranks}")
else:
    print("RRF складывает 1/(60+rank) по каждому списку. Чанк, попавший в топ обоих,")
    print("получает вклад дважды и всплывает — даже если по отдельности он вторым.")

## Ступень 2. Reranking: возвращаемся к «Грузии»

Теперь — главное. Вспомним провал из урока 4.1: вопрос «Можно ли работать из
Грузии два месяца?» приносил не тот документ. Но вот сюрприз: нужный документ
`policy-remote` **на самом деле находится** — он есть в пуле кандидатов, просто
зарыт лексическими скорами ниже топ-4. Убедимся.

In [ ]:
if QDRANT_UP:
    GEORGIA = "Можно ли работать из Грузии два месяца?"
    # широкий невод: 20 кандидатов, порог НЕ применяем (min_similarity=0.0)
    wide = retrieve(GEORGIA, employee, kb, emb, lexical=lex,
                    candidates_k=20, final_k=20, min_similarity=0.0)
    docs = [h.doc_id for h in wide.hits]
    print(f"кандидатов в неводе: {len(docs)}")
    print("policy-remote среди кандидатов:", "policy-remote" in docs)
    pos = docs.index("policy-remote") + 1 if "policy-remote" in docs else None
    print(f"...но по лексическому рангу он только на позиции: {pos}")
    print("\nТо есть recall есть — потерян precision. Это работа для reranker.")
else:
    print("Мой прогон: policy-remote ЕСТЬ среди 13 кандидатов, но зарыт вниз")
    print("лексическими скорами. Значит, поиск его нашёл — надо лишь переранжировать.")

### bi-encoder против cross-encoder

Почему обычный эмбеддер не поставил `policy-remote` наверх, а reranker поставит
(вспоминаем урок 2.6):

```
bi-encoder (обычный эмбеддинг):
   вектор("Грузия...")   ─┐
                          ├─ косинус  ← считаются НЕЗАВИСИМО, дёшево
   вектор("из-за границы")┘             можно проиндексировать заранее

cross-encoder (reranker):
   ["Грузия..." + "из-за границы..."] ─▶ модель ─▶ балл
                                          ↑ читает ПАРУ целиком, attention
                                            видит оба текста разом — точнее,
                                            но заранее ничего не посчитать
```

cross-encoder читает вопрос и чанк **вместе** и понимает, что «работать из
Грузии» и «работа из-за границы до 90 дней» — про одно и то же. Расплата: по
одному прогону модели на каждого кандидата, поэтому reranker пускают только на
20–100 отобранных кандидатов, а не на весь индекс.

### LLM как reranker — без скачивания моделей

Настоящий cross-encoder надо скачивать (~470 МБ, разберём ниже). Но у нас уже
поднята qwen — используем **её** как судью релевантности. `LLMReranker` из
`rag_service` задаёт модели один вопрос на кандидата: «оцени 0–3, насколько
фрагмент отвечает на вопрос».

In [ ]:
from rag_service.retrieval import LLMReranker

if QDRANT_UP and OLLAMA_UP:
    reranker = LLMReranker(OllamaLLM())
    # широкий невод (min_similarity=0.0!) + reranker
    r = retrieve(GEORGIA, employee, kb, emb, lexical=lex,
                 reranker=reranker, use_rerank=True,
                 candidates_k=20, final_k=4, min_similarity=0.0)
    print("Вопрос:", GEORGIA)
    print("\nПосле LLM-reranking, top-4 (колонка rerank — балл судьи 0..3):")
    print(format_hits(r.hits))
    print("\npolicy-remote на 1-м месте:", r.hits[0].doc_id == "policy-remote")
else:
    print("Мой прогон: LLM-reranker поднял policy-remote на 1-е место (балл 2),")
    print("остальные кандидаты получили 0. Семантический разрыв из урока 4.1 закрыт —")
    print("не сменой эмбеддера, а вторым проходом, который читает пару (вопрос, чанк).")

В моём прогоне reranker поставил `policy-remote` **на первое место** с
баллом 2, а всем остальным кандидатам дал 0. Тот самый провал из урока 4.1
закрыт — причём **без** смены эмбеддера: поиск нашёл документ ещё на первой
ступени, reranker просто увидел его релевантность там, где лексика не увидела.

> ⚠️ Оговорка про честность: единичные оценки маленькой модели-судьи шумят
> (мы это разбирали в уроке 3.3 — GPU при t=0 не даёт бит-в-бит повтора).
> На одном запросе балл может дрогнуть; смотрите на агрегаты, а cross-encoder
> (ниже) стабильнее LLM-судьи.


### Важная грабля: порог ДО reranking

Обратите внимание на `min_similarity=0.0` в ячейке выше — это не случайность.
Косинус `policy-remote` для этого вопроса ~0.08, а наш дефолтный порог
`min_similarity=0.10`. Если применить порог **до** reranking, нужный чанк
отсекается — и reranker его уже не увидит. Порог, задуманный как защита от
мусора, убивает recall, который должен спасти reranker.

In [ ]:
if QDRANT_UP and OLLAMA_UP:
    # тот же запрос, но с боевым порогом 0.10 ДО reranking
    r_strict = retrieve(GEORGIA, employee, kb, emb, lexical=lex,
                        reranker=reranker, use_rerank=True,
                        candidates_k=20, final_k=4, min_similarity=0.10)
    docs = [h.doc_id for h in r_strict.hits]
    print("С порогом 0.10 ДО reranking, top-4:", docs)
    print("policy-remote спасён:", "policy-remote" in docs)
    print("\nВывод: в двухступенчатой схеме держите невод ШИРОКИМ —")
    print("порог отсечения применяйте ПОСЛЕ reranking (по баллу reranker), а не до.")
else:
    print("Мой прогон: с порогом 0.10 policy-remote отсекается ДО reranker и")
    print("в top-4 не попадает. Порог до reranking убивает recall, который нужен reranker'у.")

## Настоящий cross-encoder (опционально, требует скачивания)

`LLMReranker` удобен (ничего не качать), но платит временем: по вызову LLM на
кандидата — на 20 кандидатов это десятки секунд. В проде ставят специальный
**cross-encoder**: он в разы быстрее и обучен именно на ранжирование. Наш
`Reranker` использует `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` (~470 МБ,
мультиязычный, работает на CPU).

Модель **не скачана** — ячейка ниже проверяет её наличие и мягко пропускается.
Чтобы попробовать: `pip install sentence-transformers`, и модель подтянется с
Hugging Face при первом запуске.

In [ ]:
# Настоящий cross-encoder. Скачивание — по вашей команде, не автоматически.
from rag_service.config import settings

try:
    from sentence_transformers import CrossEncoder  # noqa: F401
    from huggingface_hub import snapshot_download
    # проверяем, скачана ли модель, БЕЗ попытки её тянуть
    snapshot_download(settings.reranker_model, local_files_only=True)
    RERANKER_READY = True
except Exception:
    RERANKER_READY = False
    print("cross-encoder не скачан — это ожидаемо. Чтобы включить:")
    print("  pip install sentence-transformers")
    print(f"  # модель '{settings.reranker_model}' подтянется сама (~470 МБ)")

if RERANKER_READY and QDRANT_UP:
    from rag_service.retrieval import Reranker
    ce = Reranker()
    r = retrieve(GEORGIA, employee, kb, emb, lexical=lex,
                 reranker=ce, use_rerank=True,
                 candidates_k=20, final_k=4, min_similarity=0.0)
    print("cross-encoder top-4:", [(h.doc_id, h.rerank_score) for h in r.hits])

**LLM-судья или cross-encoder — что выбрать:**

| | LLM-reranker (наш) | cross-encoder |
|---|---|---|
| Зависимости | ничего (уже есть qwen) | +sentence-transformers, +модель ~470 МБ |
| Скорость | медленно (вызов LLM на кандидата) | быстро (батч, спец-модель) |
| Стабильность | шумит на маленькой модели | стабильнее |
| Когда | прототип, мало кандидатов | продакшен, поток запросов |

Интерфейс у них одинаковый (`rerank(query, hits, top_n)`), поэтому в
`retrieve()` они взаимозаменяемы — меняете одну строку.


## Multi-query: расширяем невод переформулировками

Reranking чинит **precision** (нужный чанк был найден, но стоял низко). А что
если чанк вообще **не попал** в невод — потому что в вопросе и документе нет ни
одного общего слова? Тогда нужен **recall**: закинуть в поиск не один запрос, а
несколько переформулировок, и объединить результаты.

`expand_query` просит LLM переписать вопрос синонимами и официальной лексикой.

In [ ]:
from rag_service.retrieval import expand_query

if OLLAMA_UP:
    variants = expand_query(GEORGIA, OllamaLLM(), n=3)
    print("Переформулировки вопроса про Грузию:")
    for v in variants:
        print("   •", v)
else:
    print("Мой прогон: qwen переформулировала «Грузию» как «временная работа за")
    print("рубежом», «за пределами страны», «краткосрочная работа». Идея верная.")

### Честный предел multi-query на маленькой модели

А теперь — честность, ради которой этот урок стоит проходить целиком. В моих
прогонах qwen переформулировала «работать из Грузии» как «за рубежом», «за
пределами страны» — по смыслу верно. Но в документе написано **«из-за границы»**,
и на нашем **лексическом** стеке (TF-IDF + BM25) эти слова всё равно не
совпадают. Multi-query поднял recall, но конкретно этот синонимический разрыв
на лексике **не закрыл** — его закрыл reranking (который читает смысл пары).

Мораль: multi-query — не панацея. Он помогает на жаргоне и коротких запросах,
но переформулировки 3B-модели могут не попасть в точную лексику документа, а
на чисто лексическом поиске synonym gap остаётся. И платит он дорого: +1 вызов
LLM и ×N прогонов поиска на каждый запрос.


## Настоящая починка recall: семантический эмбеддер e5 (опционально)

Если синонимических разрывов много — правильное решение не костыли, а
**семантический эмбеддер**. `intfloat/multilingual-e5-small` (~470 МБ) понимает,
что «Грузия», «из-за границы» и «за рубежом» — про одно, ещё на первой ступени,
без reranking и multi-query.

Наш пакет умеет переключаться на e5 одной переменной окружения — интерфейс
эмбеддера тот же (мы заложили это в уроке 4.1). Модель **не скачана**; ячейка
показывает, как включить, и мягко пропускается.

In [ ]:
# Переключение на семантический эмбеддер e5. Скачивание — по вашей команде.
#   pip install sentence-transformers
#   $env:RAG_EMB_BACKEND = "e5"                      # PowerShell
#   python -m rag_service.ingest --reset            # переиндексация под e5!
# ВАЖНО: у e5 другая размерность и другая геометрия — нужна ОТДЕЛЬНАЯ коллекция
# и свой порог (для e5 годные скоры ~0.75-0.90, а не ~0.10 как у TF-IDF).
try:
    from huggingface_hub import snapshot_download
    from rag_service.config import settings
    snapshot_download(settings.e5_model, local_files_only=True)
    E5_READY = True
    print("e5 скачан — можно переиндексировать с RAG_EMB_BACKEND=e5 и сравнить recall.")
except Exception:
    E5_READY = False
    print("e5 не скачан — это ожидаемо (правило курса: модели тянем только по команде).")
    print("Ожидаемый эффект после включения: 'работать из Грузии' находит policy-remote")
    print("уже на первой ступени, БЕЗ reranking — потому что e5 понимает синонимы.")

## Собираем конвейер: что когда включать

```
вопрос
  │
  ├─ multi-query?   только для коротких/жаргонных запросов (дорого: +LLM)
  ├─ dense (e5 лучше TF-IDF на синонимах) ─┐
  ├─ BM25 (точные строки, коды)           ─┤─ RRF ─▶ 20 кандидатов
  │                                         ┘
  ├─ reranking (cross-encoder > LLM-судья) ─▶ top-4 по баллу
  └─ порог отсечения ПОСЛЕ reranking       ─▶ мусор → отказ
```

| Приём | Чинит | Цена | Включать когда |
|---|---|---|---|
| BM25 + RRF | recall на точных словах | почти бесплатно | почти всегда |
| reranking | precision | средняя–высокая | когда важен top-1..4 |
| multi-query | recall на жаргоне | высокая (+LLM×N) | короткие/жаргонные запросы |
| e5 вместо TF-IDF | recall на синонимах | скачивание + RAM | много синонимических разрывов |

Начинайте с дешёвого (BM25+RRF), добавляйте дорогое (reranking, e5) только там,
где метрики из урока 4.3 показывают реальную проблему. Оптимизировать поиск
вслепую — верный способ потратить неделю и замедлить сервис.


## ⚠️ Частые ошибки

- **Порог отсечения ДО reranking.** Самая коварная. Тесный порог выбрасывает
  чанк, который reranker вытащил бы наверх. В двухступенчатой схеме невод —
  широкий, порог — по баллу reranker, ПОСЛЕ него.
- **reranking всего индекса.** cross-encoder — это N прогонов модели. Пускать
  его на весь индекс = убить латентность. Только на 20–100 отобранных кандидатов.
- **Нормализация скоров dense и BM25.** Косинус и BM25 несравнимы. Не нормализуйте
  их «на глаз» — сливайте по рангам (RRF).
- **multi-query на всё подряд.** Это ×N поиска и +вызов LLM на каждый запрос.
  Для точных запросов («10.10.0.53») это лишние секунды и риск размыть выдачу.
- **Смена эмбеддера без переиндексации и нового порога.** e5 и TF-IDF — разная
  размерность и разная шкала близости. Нужна отдельная коллекция и свой порог.
- **«Куплю модель побольше — поиск станет лучше».** Часто дешевле и эффективнее
  добавить BM25 и reranking, чем менять эмбеддер на тяжёлый.


## Упражнения

1. **RRF своими руками.** Возьмите два коротких списка (dense и bm25) на любом
   вопросе и посчитайте `rrf_score` для каждого чанка вручную по формуле
   `Σ 1/(60+rank)`. Сверьте с тем, что вернул `rrf_fuse`. Убедитесь, что чанк,
   стоящий 2-м в обоих списках, обгоняет чанк, стоящий 1-м только в одном.

2. **Кривая recall от невода.** Для вопроса про Грузию постройте таблицу: при
   `candidates_k` = 4, 8, 12, 20 — попадает ли `policy-remote` в невод (до
   reranking). Найдите минимальный `candidates_k`, при котором reranker уже
   способен его спасти. Это цена recall в кандидатах.

3. **LLM-судья против шума.** Прогоните `LLMReranker` на вопросе про Грузию
   3–5 раз и запишите балл `policy-remote` каждый раз. Насколько стабильна оценка
   3B-судьи? Предложите, как снизить шум (голосование, шкала 0/1 вместо 0..3 —
   вспомните урок 3.3).


## Мини-квиз

<details><summary><b>1. Чем отличаются recall и precision в поиске и почему нужны две ступени?</b></summary>

**Recall** — «не потерять нужный чанк» (полнота невода). **Precision** — «наверх
попадает только лучшее» (точность топа). Одна модель редко хороша в обоих:
быстрый поиск (эмбеддинги, BM25) оптимизирован под recall на всём индексе,
cross-encoder — под precision, но слишком медленный для всего индекса. Поэтому:
ступень 1 — широкий дешёвый невод (recall), ступень 2 — дорогой reranker
на кандидатах (precision).
</details>

<details><summary><b>2. Почему RRF складывает ранги, а не сами скоры dense и BM25?</b></summary>

Косинус (например, 0.71) и BM25 (например, 12.4) — величины из разных шкал,
честно нормализовать их нельзя. Ранги же сравнимы всегда: 1-е место в любом
списке значит одно и то же. RRF складывает `1/(k+rank)` по спискам, поэтому
чанк, попавший в топ обоих, всплывает — а несравнимость скоров не мешает.
</details>

<details><summary><b>3. Reranking «нашёл» policy-remote, которого не было в top-4. Как это возможно?</b></summary>

Он там был — просто ниже top-4, в широком неводе кандидатов. Reranking ничего
не «находит» заново, он **переупорядочивает уже найденное**. Поэтому reranker
бессилен, если нужный чанк не попал в невод вообще (это лечится широким
`candidates_k`, multi-query или лучшим эмбеддером), и бесполезен, если порог
отсёк чанк ещё до reranking.
</details>

<details><summary><b>4. Почему порог релевантности нельзя применять до reranking?</b></summary>

Потому что цель невода — recall: собрать всё, что *может* подойти, включая чанки
с низким скором первой ступени. Reranking затем поднимет из них релевантные.
Если применить порог до reranking, вы выбросите именно те низкоскоровые чанки,
которые reranker вытащил бы наверх (наш `policy-remote` с косинусом ~0.08 при
пороге 0.10). Порог — ПОСЛЕ reranking, по его баллу.
</details>

<details><summary><b>5. Multi-query переформулировал «Грузию» верно по смыслу, но recall не вырос. Почему?</b></summary>

Переформулировки («за рубежом», «за пределами страны») по смыслу правильные, но
в документе написано конкретно «из-за границы». На **лексическом** стеке
(TF-IDF/BM25) слова всё равно не совпадают — совпадение слов, а не смысла.
Multi-query помогает, когда переформулировка попадает в точную лексику
документа; чистый синонимический разрыв закрывает семантический эмбеддер (e5)
или reranking, читающий смысл пары.
</details>


## Решения

Три упражнения про две ступени поиска: как именно RRF сливает списки, сколько
recall стоит в кандидатах и насколько можно верить LLM-судье.

**Решение 1 — RRF своими руками.** Формула короткая: `score = Σ 1/(60 + rank)`
по каждому списку, где чанк встретился. Посчитаем вручную и сверим с `rrf_fuse`
на реальных списках, а потом проверим ключевое свойство RRF на чистом примере.

In [ ]:
from rag_service.retrieval import rrf_fuse
from rag_service.access import build_filter
from rag_service.store import Hit

if QDRANT_UP:
    q = "оборудование для удалённой работы"
    dense = kb.search(emb.encode_query(q), k=5, flt=build_filter(employee))
    bm = lex.search(q, k=5)
    print("dense:", [h.doc_id for h in dense])
    print("bm25 :", [h.doc_id for h in bm])

    def manual_rrf(runs, k=60):
        scores = {}
        for run in runs:
            for rank, h in enumerate(run, start=1):
                scores[h.id] = scores.get(h.id, 0.0) + 1.0 / (k + rank)
        return scores

    manual = manual_rrf([dense, bm])
    fused = rrf_fuse({"dense": dense, "bm25": bm})
    print("\nсверка ручного расчёта с rrf_fuse:")
    for h in fused[:5]:
        ok = abs(manual[h.id] - (h.rrf_score or 0)) < 1e-6
        print(f"  {h.doc_id[:18]:18s} ранги={h.ranks} rrf_fuse={h.rrf_score} "
              f"вручную={round(manual[h.id], 6)} совпало={ok}")
else:
    print("Нужен Qdrant. rrf_fuse суммирует 1/(60+rank) по каждому списку.")

# свойство RRF на чистом примере: A — 2-й в ОБОИХ, X и Y — 1-е, но каждый в одном
def _h(doc):
    return Hit(id=doc, text=doc, metadata={"doc_id": doc})
demo_dense = [_h("X_первый_только_тут"), _h("A_второй_в_обоих")]
demo_bm25 = [_h("Y_первый_только_тут"), _h("A_второй_в_обоих")]
print("\nсвойство RRF (2-й в обоих против 1-го в одном):")
for h in rrf_fuse({"dense": demo_dense, "bm25": demo_bm25}):
    print(f"  {h.doc_id:22s} rrf={h.rrf_score}  ранги={h.ranks}")

Ручной расчёт совпал с `rrf_fuse` до шестого знака — никакой магии, просто
сумма обратных рангов. А чистый пример показывает, ради чего RRF затевался:
чанк `A`, стоящий **2-м в обоих** списках, получает `1/62 + 1/62 ≈ 0.0323` и
обгоняет `X` и `Y`, каждый из которых **1-й, но только в одном** списке
(`1/61 ≈ 0.0164`). Согласие двух разных методов ценнее единоличного лидерства
одного — именно поэтому гибридный поиск устойчивее любого одиночного.

**Решение 2 — кривая recall от невода.** Reranker умеет только переставлять то,
что уже в неводе: если `policy-remote` не попал в кандидаты, никакой реранкинг
его не спасёт. Найдём минимальный `candidates_k`, при котором он вообще
появляется (порог `min_similarity=0.0`, реранкинг выключен — смотрим сырой невод):

In [ ]:
if QDRANT_UP:
    GEORGIA = "Можно ли работать из Грузии два месяца?"
    first_k = None
    print(f"{'candidates_k':>12} {'в неводе':>10} {'позиция':>9}")
    for ck in (4, 8, 12, 20):
        wide = retrieve(GEORGIA, employee, kb, emb, lexical=lex,
                        candidates_k=ck, final_k=ck, min_similarity=0.0, use_rerank=False)
        docs = [h.doc_id for h in wide.hits]
        inside = "policy-remote" in docs
        pos = docs.index("policy-remote") + 1 if inside else None
        if inside and first_k is None:
            first_k = ck
        print(f"{ck:>12} {str(inside):>10} {str(pos):>9}")
    print(f"\nминимальный candidates_k, при котором reranker способен спасти "
          f"policy-remote: {first_k}")
else:
    print("Нужен Qdrant. Идея: policy-remote входит в невод не сразу — до этого "
          "reranker бессилен, потому что переставлять нечего.")

Вот она, цена recall в кандидатах: при `candidates_k=4` нужного документа
в неводе **нет** — реранкер бессилен; он появляется только начиная с
`candidates_k=8` (лексический ранг ~8). Отсюда правило двухступенчатого поиска:
**невод держим широким** (дефолт сервиса — 20), потому что recall, потерянный на
первой ступени, вторая уже не вернёт. Дорого ли это? Дешевле, чем кажется:
расширить невод — это лишние строки из Qdrant (миллисекунды), а не лишние
вызовы LLM.

**Решение 3 — LLM-судья против шума.** Прогоним судью на `policy-remote`
многократно и посмотрим на стабильность. `LLMReranker` внутри зовёт модель с
`temperature=0.0` — сравним это с `temperature=0.8`, чтобы увидеть природу шума:

In [ ]:
import re

if QDRANT_UP and OLLAMA_UP:
    wide = retrieve(GEORGIA, employee, kb, emb, lexical=lex,
                    candidates_k=20, final_k=20, min_similarity=0.0, use_rerank=False)
    remote = next(h for h in wide.hits if h.doc_id == "policy-remote")
    llm = OllamaLLM()

    def judge(text, temperature):
        prompt = (f"Вопрос сотрудника: {GEORGIA}\n\nФрагмент документа:\n{text[:400]}\n\n"
                  "Насколько фрагмент помогает ответить на вопрос? Ответь ОДНОЙ цифрой: "
                  "0 (не по теме), 1 (косвенно), 2 (частично), 3 (прямой ответ).")
        out = llm.chat("Ты оцениваешь релевантность фрагмента документа вопросу сотрудника.",
                       prompt, temperature=temperature, max_tokens=4).text
        m = re.search(r"[0-3]", out)
        return int(m.group()) if m else 0

    for temp in (0.0, 0.8):
        runs = [judge(remote.text, temp) for _ in range(7)]
        binary = [1 if s >= 2 else 0 for s in runs]
        print(f"temp={temp}: баллы 7x = {runs}  разброс={max(runs) - min(runs)} "
              f"| бинарно (>=2 -> релевантно): {binary}")
else:
    print("Нужны Qdrant + Ollama.")

Честный результат, и он поучительнее ожидаемого. При `temperature=0`
(а именно так зовёт модель `LLMReranker`) судья **детерминирован**: одна и та же
пара (вопрос, чанк) даёт один и тот же балл, разброс за 7 прогонов — ноль. Жадное
декодирование не бросает кубик, поэтому «шума от прогона к прогону» в дефолтной
конфигурации просто нет.

Шум появляется, только если поднять температуру: при `temp=0.8` балл
`policy-remote` гуляет между 2 и 3 — «частично» против «прямого ответа». Но
приглядитесь: обе оценки означают «релевантно». Колебание есть, а разницы для
ранжирования — нет.

Отсюда три способа снизить шум, по возрастанию цены:

1. **Держать `temperature=0`** — бесплатно и уже сделано в `LLMReranker`. Если
   не гонитесь за разнообразием, это главный ответ.
2. **Голосование** (когда семплируете с `temp>0`): взять медиану N прогонов —
   медиана `[3,2,2,2,2,2,2]` = 2, случайная тройка растворяется.
3. **Огрубить шкалу до бинарной** (0/1: релевантно / нет). Колонка «бинарно» в
   выводе стабильна `[1,1,1,…]` при **обеих** температурах — потому что дрожание
   2↔3 исчезает, как только мы перестаём различать градации «релевантного». Это
   ровно урок 3.3: бинарный судья устойчивее шкалы 0..3, потому что у него проще
   граница решения. Для реранкинга, где важен лишь порядок «релевантное выше
   нерелевантного», трёх градаций и не нужно.

## Итоги

- Качественный поиск — **двухступенчатый**: широкий невод (recall) → точный
  reranker (precision). Ни одна модель не хороша в обоих сразу.
- **Гибрид dense + BM25 + RRF** покрывает слепые зоны друг друга почти бесплатно;
  сливать списки — по рангам (RRF), не по несравнимым скорам.
- **Reranking** вытащил `policy-remote` с задворков на 1-е место — тот самый
  провал из урока 4.1 закрыт вторым проходом, а не сменой эмбеддера.
- **Порог — ПОСЛЕ reranking.** Порог до него убивает recall, который reranker
  должен спасти (самая частая ошибка двухступенчатых схем).
- **multi-query** поднимает recall на жаргоне, но на лексическом стеке не
  закрывает чистый синонимический разрыв и стоит дорого (+LLM×N).
- Настоящая починка синонимов — **семантический эмбеддер e5** (опционально,
  требует скачивания); включайте дорогое только там, где метрики показывают боль.

## Что дальше

Поиск мы отточили. Осталась последняя большая тема надёжности — **кто что имеет
право найти**. Урок **4.5 «Метаданные: основа безопасности и персонализации»**:
фильтрация по правам доступа, мультитенантность и версионирование документов.
`access.py` мы уже написали — пора разобрать его до винтика.

## Полезные ссылки
- Reciprocal Rank Fusion (Cormack et al., 2009): <https://plg.uwaterloo.ca/~gvcorma/cormacksigir09-rrf.pdf>
- sentence-transformers, cross-encoders: <https://www.sbert.net/examples/applications/cross-encoder/README.html>
- multilingual-e5-small: <https://huggingface.co/intfloat/multilingual-e5-small>
- Обзор RAG-Fusion / multi-query: <https://arxiv.org/abs/2402.03367>
